# bge-reranker-v2-m3 리랭킹 실험 (KDIC RAG)

검색은 로컬 프로덕션 경로에서 이미 굳혀 올린다. 이 노트북은 **리랭킹만** 한다 —
검색이 함께 바뀌면 리랭커 효과를 분리할 수 없다.

## 왜 지금 다시 하는가

이전 리랭커 실험(`crawler/eval_rerank.py`, `Dongjin-kr/ko-reranker`)은 **문서 단위**
(`parent_doc_id ∈ gt_docs`) 판정으로 측정했고, 그래서 개선이 거의 없었다. 헤드룸 자체가
없었기 때문이다:

| 판정 입도 | hit@5 | hit@50 (천장) | 헤드룸 |
|---|---:|---:|---:|
| 문서(parent_doc_id) | .9767 | .9977 | **+.021** |
| 청크(chunk_id) | .8023 | .9209 | +.119 |
| **앵커텍스트** | **.8186** | **.9302** | **+.112** |

*(TEST 430셋, `RRF_ALPHA=0.65 · RRF_K=1` 기준)*

문서 단위로는 2.1%p 여지밖에 없었으니 "소폭 상향"이 나온 게 당연하다. 청크·앵커
단위로는 **11~16%p** 여지가 있고, 리랭커가 잘하는 일(문서 안에서 맞는 청크 고르기)이
정확히 그 구간이다.

- TEST 430: top-50엔 있는데 top-5엔 없는 문항 **48건(11.2%)** · pool에도 없어 리랭커 무관 30건(7.0%)
- VAL 355: 건질 수 있는 문항 **58건(16.3%)** · pool에도 없음 4건(1.1%)

pool은 이미 충분히 좋다. 병목은 **순서**다.


## 0. 런타임 확인

`런타임 > 런타임 유형 변경`에서 **GPU(T4 이상)** 선택. CPU로도 돌지만 수십 배 느리다.


In [ ]:
import subprocess, sys
print(sys.version)
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total',
                                   '--format=csv,noheader'], text=True).strip())
except Exception as e:
    print('GPU 없음 — CPU로 진행(느림):', type(e).__name__)


## 1. 환경 확인 — 설치하지 않는다

**torch를 업그레이드하지 말 것.** Colab은 torch와 torchvision을 버전 짝을 맞춰 깔아두는데,
torch만 올리면 짝이 어긋나 transformers의 lazy import가 이렇게 깨진다:

```
RuntimeError: operator torchvision::nms does not exist
→ ModuleNotFoundError: Could not import module 'XLMRobertaForSequenceClassification'
```

`bge-reranker-v2-m3`는 Colab 기본 torch/transformers로 그냥 돈다. 설치할 것이 없다.
이미 깨진 세션이면 **런타임 > 세션 다시 시작**으로 기본 torch로 되돌린 뒤 처음부터 다시 실행한다.


In [ ]:
import torch, transformers
print('torch       ', torch.__version__)
print('transformers', transformers.__version__)

try:
    import torchvision
    print('torchvision ', torchvision.__version__, '— torch와 짝 정상')
except Exception as e:
    print('torchvision 깨짐:', type(e).__name__, e)
    print('→ 런타임 > 세션 다시 시작 후, torch를 업그레이드하지 말고 다시 실행하세요.')

from packaging.version import Version
assert Version(transformers.__version__) >= Version('4.40'), (
    f'transformers {transformers.__version__} 가 낮습니다. '
    "pip install -q -U --no-deps transformers 로 올린 뒤 세션을 다시 시작하세요 "
    '(--no-deps 가 torch 재설치를 막습니다).')
print('\n설치 불필요 — 다음 셀로 진행')


## 2. 입력 업로드

로컬에서 먼저 만든다:
```bash
cd crawler && python ../rerank_colab/prepare_inputs.py   # → rerank_colab/rerank_inputs.zip
```
zip 내용: `chunks.jsonl` · `gt_natural430.jsonl` · `gt_natural355.jsonl` ·
`candidates_{430,355}.jsonl`(top-50 + 검색 점수/순위) · `meta.json`(파라미터 + 베이스라인 지표)


In [ ]:
import json, zipfile, io, os

if not os.path.exists('chunks.jsonl'):
    from google.colab import files
    up = files.upload()                      # rerank_inputs.zip 선택
    name = next(iter(up))
    with zipfile.ZipFile(io.BytesIO(up[name])) as z:
        z.extractall('.')

def load(p):
    with open(p, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

CHUNKS = {c['chunk_id']: c for c in load('chunks.jsonl')}
META = json.load(open('meta.json', encoding='utf-8'))
print('청크', len(CHUNKS), '· 파라미터', META['params'])
for k, v in META['baseline'].items():
    print(f"  베이스라인 {k}: 앵커 hit@5 {v['anchor']['hit@5']:.4f} · 천장 hit@50 {v['anchor']['hit@50']:.4f}")


## 3. 판정 로직 — `crawler/eval_anchor_v3.py`와 동일 규약

여기가 어긋나면 이후 수치 전부가 무의미하다. 평가기 규약을 그대로 옮긴 것:

- 앵커는 `gt_anchors`(신) → `gt_anchor_span`(구) 순으로 읽고 NFKC + 공백 전면 제거로 정규화
- 앵커가 덮지 않는 gold 청크는 `chunk_id`로 판정(하이브리드) — 앵커를 지어내지 않기 위함
- `and` = ALL(앵커 전부 + 미앵커 gold 전부), 그 외(`single`/`or`/`near_dup`/`duplicate`/`contained`) = ANY

**4번 셀에서 리랭킹 없이 베이스라인을 재현해 `meta.json`과 대조한다.** 그게 통과해야 이 노트북을 믿을 수 있다.


In [ ]:
import re, unicodedata

KS = (1, 3, 5, 10, 30, 50)
RELATION_ANY = {'single', 'or', 'duplicate', 'near_dup', 'contained'}
RELATION_ALL = {'and'}

def norm_text(t):
    return re.sub(r'\s+', '', unicodedata.normalize('NFKC', t or ''))

NT = {cid: norm_text(c.get('text', '')) for cid, c in CHUNKS.items()}

def gold_ids(it):
    ids = it.get('gt_chunk_ids') or ([it['gt_chunk_id']] if it.get('gt_chunk_id') else [])
    if isinstance(ids, str):
        ids = [ids]
    return list(dict.fromkeys(ids))

def get_anchors(it):
    a = it.get('gt_anchors')
    if isinstance(a, str):
        a = [a]
    if not a:
        s = it.get('gt_anchor_span')
        a = [s] if s else []
    return list(dict.fromkeys(norm_text(x) for x in a if x and x.strip()))

def relation(it):
    lab = it.get('chunk_relation') or 'single'
    if lab in RELATION_ALL:
        return 'and'
    if lab in RELATION_ANY:
        return 'any'
    raise ValueError(f"{it.get('id')}: unsupported chunk_relation={lab!r}")

def build_conditions(it, use_anchor=True):
    """문항 → (is_and, [조건집합...]). 집합 하나가 '이 중 하나라도 top-k면 충족'."""
    g = gold_ids(it)
    is_and = relation(it) == 'and'
    if not use_anchor:
        return is_and, ([{c} for c in g] if is_and else ([set(g)] if g else []))
    anchors = get_anchors(it)
    sets = [s for s in ({c for c in NT if a in NT[c]} for a in anchors) if s]
    covered = set().union(*sets) if sets else set()
    sets += [{c} for c in g if c not in covered]      # 앵커 없는 gold → chunk_id 폴백
    return is_and, sets

def score_run(items, orders, use_anchor=True):
    """orders: {id: [chunk_id ...]} 순위순. → hit@k · MRR@10"""
    m = {f'hit@{k}': 0 for k in KS}
    m['mrr'] = 0.0
    n = 0
    for it in items:
        if it.get('answerable') is not True:
            continue
        is_and, sets = build_conditions(it, use_anchor)
        if not sets:
            continue
        n += 1
        pos = {c: r for r, c in enumerate(orders[it['id']], 1)}
        firsts = [min((pos[c] for c in s if c in pos), default=None) for s in sets]
        if is_and:
            need = None if any(f is None for f in firsts) else max(firsts)
        else:
            ok = [f for f in firsts if f is not None]
            need = min(ok) if ok else None
        if need is None:
            continue
        for k in KS:
            if need <= k:
                m[f'hit@{k}'] += 1
        if need <= 10:
            m['mrr'] += 1.0 / need
    return {k: v / n for k, v in m.items()}, n


## 4. 베이스라인 재현 — 이 셀이 통과해야 이후 수치를 믿을 수 있다


In [ ]:
SETS = {}
for name in ('430', '355'):
    items = load(f'gt_natural{name}.jsonl')
    cands = {c['id']: c for c in load(f'candidates_{name}.jsonl')}
    SETS[name] = (items, cands)

ok = True
for name, (items, cands) in SETS.items():
    orders = {i: [x['chunk_id'] for x in c['candidates']] for i, c in cands.items()}
    for use_anchor, key in ((True, 'anchor'), (False, 'chunk_id')):
        got, n = score_run(items, orders, use_anchor)
        exp = META['baseline'][name][key]
        diff = {k: round(got[k] - exp[k], 6) for k in exp if abs(got[k] - exp[k]) > 1e-6}
        flag = 'OK' if not diff else f'!! 불일치 {diff}'
        ok &= not diff
        print(f"{name} {key:<9} n={n:<4} hit@5 {got['hit@5']:.4f} (기대 {exp['hit@5']:.4f})  {flag}")
assert ok, '판정 로직이 eval_anchor_v3와 어긋난다 — 아래 실험 진행 금지'
print('\n판정 동치 확인 완료')


## 5. 리랭커 로드

`BAAI/bge-reranker-v2-m3` — 다국어 cross-encoder(568M, XLM-RoBERTa 기반).
입력 `(query, passage)` → 단일 로짓. 로짓이 곧 순위이므로 sigmoid는 표시용일 뿐 순서를 안 바꾼다.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = 'BAAI/bge-reranker-v2-m3'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

tok = AutoTokenizer.from_pretrained(MODEL)
try:
    # transformers 4.5x부터 torch_dtype -> dtype 으로 이름이 바뀌었다(구버전 폴백 유지).
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, dtype=DTYPE)
except TypeError:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, torch_dtype=DTYPE)
except ModuleNotFoundError as e:
    raise RuntimeError(
        f'{e}\n\ntorch/torchvision 짝이 어긋난 세션입니다. '
        '런타임 > 세션 다시 시작 후 torch를 업그레이드하지 말고 다시 실행하세요.') from e

model = model.to(DEVICE).eval()
print(MODEL, '·', DEVICE, '·', DTYPE, '· 라벨수', model.config.num_labels)


In [ ]:
import time

@torch.no_grad()
def rerank_scores(query, chunk_ids, max_length=512, batch_size=32):
    """(query, 청크본문) 쌍의 리랭커 로짓. 청크 본문은 원문 그대로 넣는다(정규화 X)."""
    texts = [CHUNKS[c]['text'] for c in chunk_ids]
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok([query] * len(batch), batch, padding=True, truncation=True,
                  max_length=max_length, return_tensors='pt').to(DEVICE)
        out += model(**enc).logits.view(-1).float().cpu().tolist()
    return out

# 스모크 — 점수가 관련도 순으로 벌어지는지 눈으로 확인
items, cands = SETS['430']
probe = cands[items[0]['id']]
ids = [x['chunk_id'] for x in probe['candidates'][:5]]
t0 = time.perf_counter()
sc = rerank_scores(probe['question'], ids)
print(f"{probe['question']}  ({time.perf_counter()-t0:.2f}s)")
for c, s in sorted(zip(ids, sc), key=lambda x: -x[1]):
    print(f'  {s:+7.3f}  {c}  {CHUNKS[c]["text"][:60]}')


## 6. 전체 리랭킹

top-50 전량을 미리 채점해 캐시한다. 이후 깊이·블렌딩·cap 스윕은 재추론 없이 돈다.
430+355 = 785문항 × 50 ≈ 39k 쌍. T4에서 대략 5~10분.


In [ ]:
RR = {}          # {set: {id: {chunk_id: 리랭커 점수}}}
for name, (items, cands) in SETS.items():
    store = {}
    t0 = time.perf_counter()
    for n, it in enumerate(items, 1):
        c = cands[it['id']]
        ids = [x['chunk_id'] for x in c['candidates']]
        store[it['id']] = dict(zip(ids, rerank_scores(c['question'], ids)))
        if n % 100 == 0:
            print(f'  {name} {n}/{len(items)} · {time.perf_counter()-t0:.0f}s')
    RR[name] = store
    print(f'{name} 완료 {time.perf_counter()-t0:.0f}s')

json.dump(RR, open('rerank_scores.json', 'w'), ensure_ascii=False)
print('저장: rerank_scores.json — 내려두면 GPU 없이 로컬 재평가 가능')


## 7. 스윕 — 깊이 × 블렌딩 × post-cap

- **depth**: 상위 몇 개만 리랭크할지. 깊게 볼수록 건질 여지가 커지지만 리랭커 오류도 들어온다.
  depth 밖 후보는 원래 순서대로 뒤에 붙는다.
- **blend `w`**: `w × 리랭커z + (1-w) × 검색z`. 두 신호를 각각 z-정규화해 스케일을 맞춘다
  (검색 점수는 RRF라 스케일이 다르다). `w=1`이면 순수 리랭킹, `w=0`이면 원본.
- **post_cap**: 리랭킹 **뒤** 문서당 청크 수 제한. 이전 실험은 `CAP=1`이었는데 과했을 수 있어 함께 스윕.


In [ ]:
import statistics

def zscore(xs):
    if len(xs) < 2:
        return [0.0] * len(xs)
    m, s = statistics.mean(xs), statistics.pstdev(xs)
    return [0.0] * len(xs) if s == 0 else [(x - m) / s for x in xs]

DOC_OF = {cid: c['parent_doc_id'] for cid, c in CHUNKS.items()}

def reorder(cand, rr, depth, w, post_cap):
    head = cand['candidates'][:depth]
    tail = [x['chunk_id'] for x in cand['candidates'][depth:]]
    zr = zscore([rr[x['chunk_id']] for x in head])
    zs = zscore([x['score'] for x in head])
    fused = sorted(zip((x['chunk_id'] for x in head),
                       (w * a + (1 - w) * b for a, b in zip(zr, zs))),
                   key=lambda t: -t[1])
    order = [c for c, _ in fused]
    if post_cap:
        seen, keep = {}, []
        for c in order:
            d = DOC_OF[c]
            if seen.get(d, 0) < post_cap:
                keep.append(c)
                seen[d] = seen.get(d, 0) + 1
        kept = set(keep)
        order = keep + [c for c in order if c not in kept]
    return order + tail

DEPTHS = (10, 20, 30, 50)
WS = (0.0, 0.3, 0.5, 0.7, 1.0)
CAPS = (0, 1, 2, 3)          # 0 = post-cap 없음

rows = []
for name, (items, cands) in SETS.items():
    base = META['baseline'][name]['anchor']
    for depth in DEPTHS:
        for w in WS:
            for cap in CAPS:
                orders = {i: reorder(c, RR[name][i], depth, w, cap) for i, c in cands.items()}
                m, n = score_run(items, orders, use_anchor=True)
                rows.append(dict(set=name, depth=depth, w=w, cap=cap, n=n,
                                 **{k: round(m[k], 4) for k in ('hit@1','hit@3','hit@5','hit@10','mrr')},
                                 d5=round(m['hit@5'] - base['hit@5'], 4)))
    print(f'{name} 스윕 완료')

json.dump(rows, open('sweep_results.json', 'w'), ensure_ascii=False, indent=1)
print(len(rows), 'configs')


### 결과 — 두 셋에서 함께 개선되는 설정만 채택할 것


In [ ]:
def top(name, k=10):
    r = sorted((x for x in rows if x['set'] == name), key=lambda x: -x['hit@5'])
    print(f"[{name}] 베이스라인 앵커 hit@5 {META['baseline'][name]['anchor']['hit@5']:.4f}")
    print(f"{'depth':>6}{'w':>5}{'cap':>5}{'hit@1':>8}{'hit@3':>8}{'hit@5':>8}{'hit@10':>8}{'mrr':>8}{'Δhit@5':>9}")
    for x in r[:k]:
        print(f"{x['depth']:>6}{x['w']:>5}{x['cap']:>5}{x['hit@1']:>8.4f}{x['hit@3']:>8.4f}"
              f"{x['hit@5']:>8.4f}{x['hit@10']:>8.4f}{x['mrr']:>8.4f}{x['d5']:>+9.4f}")
    print()

top('430'); top('355')

# 전이성 — 두 셋 모두 좋은 설정(min 최대화). 한쪽만 좋은 값은 과적합이다.
key = lambda x: (x['depth'], x['w'], x['cap'])
A = {key(x): x for x in rows if x['set'] == '430'}
B = {key(x): x for x in rows if x['set'] == '355'}
joint = sorted((k for k in A if k in B), key=lambda k: -min(A[k]['hit@5'], B[k]['hit@5']))
print(f"{'(depth,w,cap)':<18}{'430':>9}{'355':>9}{'min':>9}{'Δ430':>9}{'Δ355':>9}")
for k in joint[:10]:
    print(f"{str(k):<18}{A[k]['hit@5']:>9.4f}{B[k]['hit@5']:>9.4f}"
          f"{min(A[k]['hit@5'], B[k]['hit@5']):>9.4f}{A[k]['d5']:>+9.4f}{B[k]['d5']:>+9.4f}")


## 8. 결과 내려받기

`rerank_scores.json`을 로컬 `rerank_colab/`에 두면 GPU 없이 재평가·재스윕할 수 있다.
`sweep_results.json`은 채택 근거로 커밋한다.


In [ ]:
from google.colab import files
files.download('rerank_scores.json')
files.download('sweep_results.json')


## 판정 시 주의

1. **한쪽 셋만 좋은 설정은 버린다.** 파라미터 튜닝에서 이미 겪었다 — TEST(430) 단독 최적은
   VAL(355)을 깎았고, VAL 단독 최적은 TEST를 베이스라인 아래로 떨어뜨렸다
   (`DECISIONS.md` 2026-08-14 ④).
2. **`hit@50`은 리랭킹으로 안 변한다** — 집합은 그대로고 순서만 바뀐다. 천장은 .9302(430) /
   .9887(355)이며, 이걸 넘기려면 검색 단계를 손봐야 한다.
3. **두 셋의 절대 수치를 나란히 놓고 회귀를 판단하지 말 것.** VAL은 `and` 문항 0건,
   TEST는 `and` 55건이라 구성이 다르다. 각각의 시계열로만 본다.
4. 최적점이 스윕 **경계**(depth=50 · w=1.0 · cap=0 등)에 붙으면 범위를 넓혀 재실행.
   경계 접촉은 최적이 아니라 그리드가 좁다는 신호다.
